# Simple Colab LLM + Open WebUI

A minimal setup to:
1. Pick an open LLM
2. Serve it with Ollama on the Colab VM
3. Open a nice chat UI (Open WebUI) accessible via a public URL

**Before running:**
- Runtime → Change runtime type → GPU (any of T4 / L4 / A100 / H100 / G4 — bigger = bigger models)
- Run cells top to bottom
- Last cell prints the public URL. Open it in any browser.

Everything lives on the Colab VM and dies when the session dies. This is the "play around" version — no Drive persistence, no domain setup, no aider. For the full setup, see the longer plan.

---

## 1. Check GPU

Confirm we have a GPU and see how much VRAM. This decides which models you can comfortably run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Pick your model

Set `MODEL` to one of the options below based on your GPU's VRAM. All are Q4 quantized (good quality, small footprint). Sizes are approximate download sizes.

| Model | VRAM needed | Download | Notes |
|---|---|---|---|
| `qwen3:8b` | ~6 GB | ~5 GB | Fits on T4 (16GB). Fast, decent quality. |
| `qwen3:14b` | ~10 GB | ~9 GB | Fits on L4 (24GB). Solid all-rounder. |
| `gemma4:27b` | ~18 GB | ~17 GB | Google Gemma 4, multimodal-capable. Strong reasoning. Needs L4+. |
| `qwen3:30b-a3b` | ~18 GB | ~17 GB | MoE — only 3B active, very fast. Needs L4+ or A100. **Recommended.** |
| `qwen3-coder:30b` | ~18 GB | ~17 GB | Coding-specialized version of above. |
| `qwen3:32b` | ~20 GB | ~19 GB | Dense 32B. Slower than 30b-a3b, similar quality. |
| `deepseek-v3` | ~130 GB | ~130 GB | DeepSeek V3 671B MoE. Needs A100 80GB ×2 or H100+. |
| `kimi-k2` | ~140 GB | ~130 GB | Moonshot Kimi K2 (1T MoE, ~32B active params). Needs H100 ×2+. |
| `qwen3:235b-a22b` | ~140 GB | ~130 GB | Needs G4 (96GB) — close fit, may OOM. |

> **Sunset:** `gemma3:12b` (superseded by Gemma 4), `llama3.3:70b` (superseded by newer generations).

Browse all models: https://ollama.com/library

In [ ]:
MODEL = "qwen3:14b"  # ← change this to whatever you want from the table above

# Optional: set a password for the Open WebUI public URL.
# You'll create the admin account on first login; this just helps you remember it.
# Anyone with the public URL can sign up unless you change WEBUI_AUTH below.
print(f"Will use model: {MODEL}")

## 3. Install Ollama

Ollama is the easiest way to run an LLM. One-line installer.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the Ollama server in the background.
import subprocess, time, os, requests

# Tell Ollama to bind to all interfaces so Open WebUI (also local) can reach it cleanly.
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait for it to be ready
for i in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("✓ Ollama is running")
        break
    except Exception:
        time.sleep(1)
else:
    print("✗ Ollama failed to start — check /tmp/ollama.log")

## 4. Download the model

First time you run this for a given model it'll take 1-10 minutes depending on size and bandwidth. Re-running with the same model is instant (already cached).

You'll see a progress bar.

In [ ]:
!ollama pull {MODEL}

In [ ]:
# Quick smoke test — make sure the model actually responds
import requests, json

r = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL, "prompt": "Say hi in 5 words.", "stream": False},
    timeout=120,
)
print(r.json().get("response", "<no response>"))

## 5. Install and start Open WebUI

ChatGPT-style interface. We install via pip (simpler than Docker in Colab) and point it at the local Ollama server.

In [ ]:
!pip install -q open-webui

In [ ]:
import subprocess, os, time, requests

env = os.environ.copy()
env["OLLAMA_BASE_URL"] = "http://localhost:11434"
env["WEBUI_AUTH"] = "True"           # require login (recommended since URL will be public)
env["ENABLE_SIGNUP"] = "True"        # so you can create your account on first visit
env["DEFAULT_USER_ROLE"] = "admin"   # first user becomes admin
env["ENABLE_OLLAMA_API"] = "True"
env["ANONYMIZED_TELEMETRY"] = "False"
env["PORT"] = "8080"

webui_proc = subprocess.Popen(
    ["open-webui", "serve", "--port", "8080"],
    env=env,
    stdout=open("/tmp/openwebui.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait for it (first start takes ~30-60s while it builds the SQLite DB)
print("Waiting for Open WebUI to start (this can take up to a minute)...")
for i in range(90):
    try:
        r = requests.get("http://localhost:8080", timeout=2)
        if r.status_code == 200:
            print("✓ Open WebUI is running on port 8080")
            break
    except Exception:
        pass
    time.sleep(1)
    if i % 10 == 9:
        print(f"  still waiting... ({i+1}s)")
else:
    print("✗ Open WebUI didn't start — check /tmp/openwebui.log")
    !tail -30 /tmp/openwebui.log

## 6. Expose it with a public URL

Cloudflare Tunnel — free, no signup needed for the quick-tunnel version. You'll get a random URL each session. (For a stable URL on your own domain, use the longer setup plan.)

In [ ]:
# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
import subprocess, re, time

# Start the tunnel — output goes to a log, we'll parse the URL from it
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8080", "--no-autoupdate"],
    stdout=open("/tmp/tunnel.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait for the URL to show up in the log
public_url = None
for i in range(30):
    time.sleep(1)
    try:
        with open("/tmp/tunnel.log") as f:
            log = f.read()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
        if m:
            public_url = m.group(0)
            break
    except Exception:
        pass

if public_url:
    print("=" * 60)
    print("  Open WebUI is live at:")
    print(f"  {public_url}")
    print("=" * 60)
    print("\nFirst visit: create your admin account (any email/password — it's local to this session).")
    print(f"Selected model in the UI: {MODEL}")
else:
    print("✗ Couldn't get a tunnel URL — check /tmp/tunnel.log")
    !tail -30 /tmp/tunnel.log

## 7. Keep this notebook running

**Don't close the Colab tab.** Everything (Ollama, Open WebUI, the tunnel) is tied to this session. Closing the tab or letting Colab idle out will kill it.

When you're done, run the shutdown cell at the bottom, or just disconnect the runtime.

---

## Troubleshooting

**Open WebUI shows no models** → It hasn't seen Ollama yet. In the UI: top-right gear icon → Settings → Connections → Ollama → make sure it's `http://localhost:11434` and click the refresh icon.

**Model is slow / OOMs** → You picked a model too big for your GPU. Restart runtime and pick a smaller one from the table.

**Tunnel URL stops working** → Re-run cell 6 (the second one). The Cloudflare quick-tunnel can occasionally drop.

**Want to add another model** → Just run `!ollama pull <model-name>` in a new cell. It'll show up in Open WebUI's model picker.

**Check logs:**
```
!tail -50 /tmp/ollama.log
!tail -50 /tmp/openwebui.log
!tail -50 /tmp/tunnel.log
```

## Optional: clean shutdown

In [ ]:
# Run this when you're done, or just close the tab
for p, name in [(tunnel_proc, "tunnel"), (webui_proc, "open-webui"), (ollama_proc, "ollama")]:
    try:
        p.terminate()
        p.wait(timeout=5)
        print(f"✓ stopped {name}")
    except Exception as e:
        print(f"  {name}: {e}")